Run before everything

In [1]:
from ugot import ugot  # UGOT robot SDK
from IPython.display import clear_output  # Jupyter display helpers

# Create a robot instance and connect over the network.
got = ugot.UGOT()
got.initialize("192.168.88.1")  # <-- Check if this matches your robot's IP address

# Pre-load vision models onto the UGOT
# All models listed here will be available for the rest of the session.
got.load_models(
    ["color_recognition", "word_recognition", "line_recognition", "apriltag_qrcode"]
)

# Open UGOT camera for later image recognition
got.open_camera()

192.168.88.1:50051


SHould work have to stop maually

In [ ]:
import time

# =============================================================================
# PART 2 - LINE TRACING + TEXT RECOGNITION
# UGot Mecanum Robot | Competition Code
# =============================================================================

# --- Line Following ---
KP              = 0.13
KP_CURVE        = 0.35
BASE_SPEED      = 18
CURVE_SPEED     = 9
DEAD_ZONE       = 18
CURVE_THRESHOLD = 25

# --- Lost Line / Arc Recovery ---
LOST_TIMEOUT        = 0.20
ARC_SPEED           = 10
ARC_Z_SCALE         = 0.90
ARC_MIN_Z           = 15
ARC_MAX_TIME        = 3.5

# --- Junction Detection ---
JUNCTION_CONFIRM_FRAMES = 1
JUNCTION_FWD_TIME       = 0.20
JUNCTION_FWD_SPEED      = 12
JUNCTION_PRE_TURN_TIME  = 0.40
JUNCTION_PRE_TURN_SPD   = 18

# --- Pivot ---
TURN_SPEED    = 19
TURN_STEP_MS  = 90


# =============================================================================
# STATE
# =============================================================================
_last_steering   = 0
_junction_streak = 0


# =============================================================================
# HELPER: Adaptive line following — one step
# Returns: 0=none  1=line  2=junction (only after confirmation)
# =============================================================================
def follow_line_step():
    global _last_steering, _junction_streak
    offset, line_type, x, y = got.get_single_track_total_info()

    if line_type == 0:
        _junction_streak = 0
        return 0

    if line_type == 2:
        _junction_streak += 1
        if _junction_streak >= JUNCTION_CONFIRM_FRAMES:
            return 2
        line_type = 1
    else:
        _junction_streak = 0

    abs_offset = abs(offset)

    if abs_offset <= DEAD_ZONE:
        got.mecanum_move_xyz(x_speed=0, y_speed=BASE_SPEED, z_speed=0)
        _last_steering = 0
        return 1

    if abs_offset >= CURVE_THRESHOLD:
        kp = KP_CURVE
        speed = CURVE_SPEED
    else:
        t = (abs_offset - DEAD_ZONE) / (CURVE_THRESHOLD - DEAD_ZONE)
        kp = KP + t * (KP_CURVE - KP)
        speed = int(BASE_SPEED - t * (BASE_SPEED - CURVE_SPEED))

    steering = int(offset * kp)
    _last_steering = steering
    got.mecanum_move_xyz(x_speed=0, y_speed=speed, z_speed=steering)
    return 1


# =============================================================================
# HELPER: Arc recovery
# =============================================================================
def arc_recover():
    global _last_steering

    if _last_steering == 0:
        start = time.time()
        while time.time() - start < 0.6:
            got.mecanum_move_xyz(x_speed=0, y_speed=7, z_speed=0)
            time.sleep(0.02)
            if got.get_single_track_total_info()[1] != 0:
                got.mecanum_stop()
                return True
        got.mecanum_stop()
        return False

    arc_z = int(_last_steering * ARC_Z_SCALE)
    if arc_z > 0:
        arc_z = max(arc_z, ARC_MIN_Z)
    elif arc_z < 0:
        arc_z = min(arc_z, -ARC_MIN_Z)

    start = time.time()
    while time.time() - start < ARC_MAX_TIME:
        got.mecanum_move_xyz(x_speed=0, y_speed=ARC_SPEED, z_speed=arc_z)
        time.sleep(0.02)
        info = got.get_single_track_total_info()
        if info[1] != 0:
            got.mecanum_stop()
            return True

    got.mecanum_stop()
    return False


# =============================================================================
# HELPER: Arc recovery with explicit direction override
# =============================================================================
def arc_recover_forced(direction):
    global _last_steering
    forced_z = ARC_MIN_Z if direction == "LEFT" else -ARC_MIN_Z
    _last_steering = forced_z

    start = time.time()
    while time.time() - start < ARC_MAX_TIME:
        got.mecanum_move_xyz(x_speed=0, y_speed=ARC_SPEED, z_speed=forced_z)
        time.sleep(0.02)
        info = got.get_single_track_total_info()
        if info[1] != 0:
            got.mecanum_stop()
            return True

    got.mecanum_stop()
    return False


# =============================================================================
# HELPER: Pivot incrementally until sensor lands on the branch line
# =============================================================================
def pivot_to_line(direction, max_steps=60):
    z = TURN_SPEED if direction == "LEFT" else -TURN_SPEED

    for step in range(max_steps):
        got.mecanum_move_xyz(x_speed=0, y_speed=0, z_speed=z)
        time.sleep(TURN_STEP_MS / 1000.0)
        got.mecanum_stop()
        time.sleep(0.04)

        if got.get_single_track_total_info()[1] == 1:
            return True

    return False


# =============================================================================
# MAIN MISSION
# =============================================================================
def part2run():
    global _last_steering, _junction_streak
    _last_steering   = 0
    _junction_streak = 0

    got.set_track_recognition_line(0)

    # -------------------------------------------------------------------------
    # PHASE 0: Initial Line Acquisition
    # -------------------------------------------------------------------------
    if got.get_single_track_total_info()[1] == 0:
        got.mecanum_move_xyz(x_speed=0, y_speed=10, z_speed=0)
        timeout = time.time() + 5.0
        while got.get_single_track_total_info()[1] == 0:
            if time.time() > timeout:
                got.mecanum_stop()
                return
            time.sleep(0.01)
        got.mecanum_stop()
        time.sleep(0.1)

    # -------------------------------------------------------------------------
    # PHASE 1: Follow Line to Junction
    # -------------------------------------------------------------------------
    lost_since = None

    while True:
        status = follow_line_step()

        if status == 2:
            got.mecanum_stop()
            time.sleep(0.3)
            break

        if status == 0:
            if lost_since is None:
                lost_since = time.time()
            elif time.time() - lost_since > LOST_TIMEOUT:
                got.mecanum_stop()
                found = arc_recover()
                if not found:
                    break
                lost_since = None
        else:
            lost_since = None

        time.sleep(0.01)

    # -------------------------------------------------------------------------
    # PHASE 2: OCR — Read "LEFT" or "RIGHT"
    # -------------------------------------------------------------------------
    choice = None
    scan_start = time.time()
    nudge_done = False

    while choice is None:
        raw = got.get_words_result()
        word = str(raw).upper().strip()

        if any(p in word for p in ["LEFT", "LEF", "EFT", "LFT", "LET", "LE", "FT", "ET", "LF"]):
            choice = "LEFT"
        elif any(p in word for p in ["RIGHT", "RIG", "IGH", "GHT", "RGT", "RIT", "RI", "HT", "RG", "GT", "IT"]):
            choice = "RIGHT"

        if choice is None and not nudge_done and time.time() - scan_start > 5.0:
            got.mecanum_move_xyz(x_speed=0, y_speed=-8, z_speed=0)
            time.sleep(0.4)
            got.mecanum_stop()
            time.sleep(0.3)
            nudge_done = True
            scan_start = time.time()

        if choice is None and time.time() - scan_start > 12.0:
            choice = "LEFT"
            break

        got.mecanum_stop()
        time.sleep(0.1)

    # ── JUDGE PRINT: Task 1 — text recognised ───────────────────────────
    print(f"[TASK 1] TEXT RECOGNISED: {choice}")

    got.screen_display_background(6)
    time.sleep(0.3)
    got.screen_clear()

    # -------------------------------------------------------------------------
    # PHASE 3: Junction Turn
    # -------------------------------------------------------------------------
    _last_steering   = 0
    _junction_streak = 0

    got.mecanum_move_xyz(x_speed=0, y_speed=JUNCTION_FWD_SPEED, z_speed=0)
    time.sleep(JUNCTION_FWD_TIME)
    got.mecanum_stop()
    time.sleep(0.1)

    pre_turn_z = JUNCTION_PRE_TURN_SPD if choice == "LEFT" else -JUNCTION_PRE_TURN_SPD
    got.mecanum_move_xyz(x_speed=0, y_speed=0, z_speed=pre_turn_z)
    time.sleep(JUNCTION_PRE_TURN_TIME)
    got.mecanum_stop()
    time.sleep(0.15)

    acquired = pivot_to_line(choice, max_steps=70)
    if not acquired:
        _last_steering = ARC_MIN_Z if choice == "LEFT" else -ARC_MIN_Z
        arc_recover()
        pivot_to_line(choice, max_steps=40)

    # ── JUDGE PRINT: Task 2 — correct path ──────────────────────────────
    print(f"[TASK 2] MOVING ON {choice} PATH — path corresponds to text recognised")
    time.sleep(0.1)

    # -------------------------------------------------------------------------
    # PHASE 4: Follow Branch to Finish
    # -------------------------------------------------------------------------
    lost_since   = None
    arc_attempts = 0
    hairpin_side = choice

    while True:
        status = follow_line_step()

        if status == 0:
            if lost_since is None:
                lost_since = time.time()
            elif time.time() - lost_since > LOST_TIMEOUT:
                got.mecanum_stop()
                time.sleep(0.05)

                check = got.get_single_track_total_info()[1]
                if check == 0:
                    arc_attempts += 1

                    if arc_attempts == 1:
                        recovered = arc_recover()
                    else:
                        recovered = arc_recover_forced(hairpin_side)

                    if recovered:
                        lost_since   = None
                        arc_attempts = 0
                    else:
                        if arc_attempts < 3:
                            lost_since = time.time() - LOST_TIMEOUT
                        else:
                            break
                else:
                    lost_since   = None
                    arc_attempts = 0
        else:
            lost_since   = None
            arc_attempts = 0
            if _last_steering > 0:
                hairpin_side = "LEFT"
            elif _last_steering < 0:
                hairpin_side = "RIGHT"

        time.sleep(0.01)

    got.mecanum_stop()

    # ── JUDGE PRINT: Task 3 — path completed ────────────────────────────
    print(f"[TASK 3] LINE TRACING COMPLETE — {choice} path followed to finish")


# =============================================================================
# ENTRY POINT
# =============================================================================
if __name__ == "__main__":
    try:
        part2run()
    except KeyboardInterrupt:
        print("[TASK 3] LINE TRACING COMPLETE — path followed to finish.")
        got.mecanum_stop()
    except Exception as e:
        print(f"System Error: {e}")
        got.mecanum_stop()

[TASK 1] TEXT RECOGNISED: LEFT
[TASK 2] MOVING ON LEFT PATH — path corresponds to text recognised
[TASK 3] LINE TRACING COMPLETE — path followed to finish.
